<a href="https://colab.research.google.com/github/Lupo6789/tesi_scacchi/blob/main/03_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

I meta dati sono i rating di bianco e nero, la differenza ello, il tempo a disposizione stimato e il flag rateb

In [2]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings

# Disabilito i warning per pulizia dell'output
warnings.filterwarnings('ignore')

print("1. CALCOLO BASELINE DINAMICHE CON METADATI COMPLETI\n")

soglie = [10, 20, 30, 40]
risultati = []

for N in soglie:
    # Carico i dataset specifici per il livello N
    df_train = pd.read_csv(f'train_features_N{N}.csv')
    df_test = pd.read_csv(f'test_features_N{N}.csv')

    # Seleziono tutti i metadati (nessuna informazione dalla scacchiera)
    features_metadata = ['white_rating', 'black_rating', 'rated', 'tempo_stimato_sec']

    X_train = df_train[features_metadata].copy()
    X_test = df_test[features_metadata].copy()

    # Calcolo e aggiungo la differenza Elo come ulteriore feature
    X_train['rating_diff'] = X_train['white_rating'] - X_train['black_rating']
    X_test['rating_diff'] = X_test['white_rating'] - X_test['black_rating']

    # Definisco il target
    y_train = df_train['winner']
    y_test = df_test['winner']

    # Modello 1: Dummy Classifier
    # Prevede sempre l'esito più frequente
    dummy = DummyClassifier(strategy='most_frequent')
    dummy.fit(X_train, y_train)
    acc_dummy = accuracy_score(y_test, dummy.predict(X_test))

    # Modello 2: Regressione Logistica
    # Addestra il modello su tutti i metadati pre-partita
    log_reg = LogisticRegression(random_state=42, max_iter=1000)
    log_reg.fit(X_train, y_train)
    acc_log = accuracy_score(y_test, log_reg.predict(X_test))

    # Salvo i risultati per costruire la tabella
    risultati.append({
        'N_semimosse': N,
        'Partite_Test': len(y_test),
        'Dummy_Accuracy (%)': round(acc_dummy * 100, 2),
        'LogReg_Accuracy (%)': round(acc_log * 100, 2)
    })

print("2. TABELLA RIASSUNTIVA DELLE BASELINE\n")
# Stampo la tabella finale
df_baseline = pd.DataFrame(risultati)
print(df_baseline.to_string(index=False))

1. CALCOLO BASELINE DINAMICHE CON METADATI COMPLETI

2. TABELLA RIASSUNTIVA DELLE BASELINE

 N_semimosse  Partite_Test  Dummy_Accuracy (%)  LogReg_Accuracy (%)
          10          3521               52.09                64.98
          20          3340               51.83                64.76
          30          3018               51.66                64.48
          40          2617               51.32                64.20
